<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

The skeleton may not have this section, but because your Colab runtime starts at /content, you need to load your GitHub repository before any other code.

In [7]:
# ML-05 setup: clone the repository and load the dataset

import os
import pandas as pd
import numpy as np

REPO_DIR = "/content/AIML"
DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"

# Clone the repository if it is not already available
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/vaishnavikabbe/AIML.git /content/AIML

# Load the dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 30000
Columns: 44


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

For the first version of the feature vector, I will use three page-level signals: `search_volume`, `impressions_90d`, and `word_count`.

These features represent search demand, recent search visibility, and content size. Missing numeric values will be filled using the median of each feature. The target `trend_direction` is kept separate from the feature vector so that the model does not receive the label as an input.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 setup: clone the repository and load the dataset

# ML-05 Section 1: Build the feature vector

feature_columns = [
    "search_volume",
    "impressions_90d",
    "word_count"
]

target_column = "trend_direction"

# Confirm required columns exist
missing_columns = [
    col for col in feature_columns + [target_column]
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Create feature dataframe
X = df[feature_columns].copy()

# Convert features to numeric
for col in feature_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Fill missing values with median
for col in feature_columns:
    X[col] = X[col].fillna(X[col].median())

# Keep target separate
y = df[target_column].copy()

print("Feature vector created successfully.")
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nMissing values after filling:")
print(X.isna().sum())


Feature vector created successfully.
Feature matrix shape: (30000, 3)
Target shape: (30000,)

Features:
['search_volume', 'impressions_90d', 'word_count']

Missing values after filling:
search_volume      0
impressions_90d    0
word_count         0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

**search_volume:** Represents the search-demand signal available for a page. Missing values are replaced with the median. It is intended to be available before the refresh-prioritization decision.

**impressions_90d:** Represents the page's observed impressions over a 90-day window. Missing values are replaced with the median. It is treated as an available historical performance signal for prioritization.

**word_count:** Represents the amount of text/content associated with the page. Missing values are replaced with the median. It is treated as a page-level content signal available before the decision.

All three features are numeric, so no categorical encoding is required for this first feature vector.

The target `trend_direction` is not included as a feature because it is the outcome the model is intended to learn or use for evaluation.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one —# ML-05 Section 2: Verify feature types, missing values, and availability

print("=== FEATURE NOTES ===")

for col in feature_columns:
    print(f"\nFeature: {col}")
    print("Data type:", X[col].dtype)
    print("Missing values:", X[col].isna().sum())
    print("Available rows:", X[col].notna().sum())

print("\n=== TARGET SEPARATION ===")
print("Target column:", target_column)
print("Target included in X:", target_column in X.columns)

=== FEATURE NOTES ===

Feature: search_volume
Data type: float64
Missing values: 0
Available rows: 30000

Feature: impressions_90d
Data type: int64
Missing values: 0
Available rows: 30000

Feature: word_count
Data type: float64
Missing values: 0
Available rows: 30000

=== TARGET SEPARATION ===
Target column: trend_direction
Target included in X: False


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I checked whether the feature vector contains the target label or obvious label-derived fields.

The target `trend_direction` is not included in the feature matrix. I also avoid using fields that represent future outcomes or information that would only become available after the refresh decision.

The 90-day impressions field is treated as a historical observation available at the decision point, rather than as a future outcome. I will continue to check the timing of features as the project develops.

No private client identifiers, URLs, domains, titles, or queries are intentionally included in the feature vector.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 Section 3: Leakage checks

print("=== LEAKAGE CHECK ===")

# 1. Check whether target is accidentally included
target_in_features = target_column in X.columns

print("Target included in feature vector:", target_in_features)

# 2. Look for suspicious column names
suspicious_keywords = [
    "label",
    "target",
    "outcome",
    "cheat",
    "future",
    "next"
]

suspicious_columns = [
    col for col in X.columns
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("\nSuspicious feature names:")
print(suspicious_columns if suspicious_columns else "None found")

# 3. Check feature matrix for missing values
print("\nMissing values in feature matrix:")
print(X.isna().sum())

# 4. Check that the target is separate
print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(target_column)

print("\nLeakage check complete.")

=== LEAKAGE CHECK ===
Target included in feature vector: False

Suspicious feature names:
None found

Missing values in feature matrix:
search_volume      0
impressions_90d    0
word_count         0
dtype: int64

Feature columns:
['search_volume', 'impressions_90d', 'word_count']

Target column:
trend_direction

Leakage check complete.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

**trend_direction:** Excluded from the feature vector because it is the target/outcome.

**Identifiers:** Excluded because identifiers do not provide useful predictive information and may create privacy risks.

**URLs, domains, titles, and private queries:** Excluded to protect privacy and because they are not necessary for this initial feature vector.

**Future outcome fields:** Excluded because information that becomes available only after the decision would create data leakage.

**Fields unrelated to the content-refresh decision:** Excluded from the first model to keep the feature vector small, interpretable, and easy to audit.

The initial feature vector therefore uses only `search_volume`, `impressions_90d`, and `word_count`.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 Section 4: Record the excluded fields and reasons

excluded_fields = {
    "trend_direction": "Target/outcome; including it would cause label leakage.",
    "Identifiers": "Excluded for privacy and because they are not useful predictive signals.",
    "URLs/domains/titles/private queries": "Excluded for privacy and because they are unnecessary for the initial model.",
    "Future outcome fields": "Excluded because they would not be available at prediction time.",
    "Unrelated fields": "Excluded to keep the first feature vector focused and interpretable."
}

print("=== EXCLUDED FIELDS AND REASONS ===")

for field, reason in excluded_fields.items():
    print(f"\n{field}:")
    print(reason)

print("\n=== FINAL FEATURE VECTOR ===")
print(X.columns.tolist())
print("Number of features:", X.shape[1])

=== EXCLUDED FIELDS AND REASONS ===

trend_direction:
Target/outcome; including it would cause label leakage.

Identifiers:
Excluded for privacy and because they are not useful predictive signals.

URLs/domains/titles/private queries:
Excluded for privacy and because they are unnecessary for the initial model.

Future outcome fields:
Excluded because they would not be available at prediction time.

Unrelated fields:
Excluded to keep the first feature vector focused and interpretable.

=== FINAL FEATURE VECTOR ===
['search_volume', 'impressions_90d', 'word_count']
Number of features: 3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.